# KASA-42 — Kaggle smoke test

**Purpose: break things here so they cannot break on the H200.**

This runs the *same* `src/kasa42/` code the H200 will run. Nothing here is a
special Kaggle version — only the numbers passed in differ (fewer languages,
smaller batches, fewer steps). Every bug this finds is a bug that would
otherwise have eaten part of a 48-hour window.

Two differences from Thursday, both handled automatically:

| | Kaggle T4 / P100 | H200 |
|---|---|---|
| Mixed precision | **fp16** + GradScaler | **bf16** |
| VRAM | 16 GB | 141 GB |

`train.py` calls `torch.cuda.is_bf16_supported()` and picks. T4 and P100 are
pre-Ampere and have no bf16, so a hardcoded bf16 would fail here — that is
exactly the class of bug this notebook exists to surface.

**Before running:** Settings → Accelerator → **GPU T4 x2**, and Internet → **On**.

## 0 · Get the code

Use the git clone if the repo is pushed; otherwise upload `src/` as a Kaggle Dataset and use the fallback.

In [ ]:
import os, sys, subprocess, pathlib

REPO_URL = 'https://github.com/YOURUSER/kasa42.git'   # <-- set this
WORK = pathlib.Path('/kaggle/working/kasa42')

if not WORK.exists():
    r = subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(WORK)],
                       capture_output=True, text=True)
    print(r.stdout or '', r.stderr or '')

if not WORK.exists():
    # Fallback: added the code as a Kaggle Dataset named 'kasa42-src'
    import shutil
    src = pathlib.Path('/kaggle/input/kasa42-src')
    assert src.exists(), 'Push to GitHub, or add src/ as a Kaggle Dataset called kasa42-src'
    shutil.copytree(src, WORK)

os.chdir(WORK)
sys.path.insert(0, str(WORK / 'src'))
print('cwd', os.getcwd())
print(sorted(p.name for p in (WORK / 'src' / 'kasa42').iterdir()))

In [ ]:
# Kaggle ships torch already; only add what is missing.
!pip install -q -U transformers datasets soundfile librosa jiwer onnx onnxruntime 2>&1 | tail -3
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('bf16 supported:', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else 'n/a',
      '-> expect False on T4/P100, True on H200')

## 1 · Logic checks

Runs the full pipeline test: real audio decode, DONDO load, blank index 33,
CTC head transfer, forward/backward, ONNX. Fix anything red before moving on.

In [ ]:
!python tests/test_text.py

In [ ]:
!python tests/test_pipeline.py

## 2 · GPU benchmark → the H200 estimate

Measures throughput in **seconds of audio per second of wall clock**, which
scales across GPUs far better than step counts, and finds the largest batch
that fits. Extrapolates a crude H200 figure.

If the estimate comes back above ~12 h, lower `--budget-hours` in `mixture.py`
or drop an epoch **now**, not on Thursday.

In [ ]:
!python tests/bench_gpu.py --minutes 4 --batch-durations 60 120 240 320

## 3 · A real (tiny) training run

Same `train()` the H200 calls. Five languages spanning the size range —
Asante Twi (200 h) down to Mampruli (~5 h) — so the mixture and bucketing meet
the same imbalance they will on Thursday.

Needs the parquet for those configs; ~2 GB, well inside Kaggle's disk.

In [ ]:
SMOKE_LANGS = ['Kusaal_kus', 'Asante_Twi_twi', 'Ewe_ewe', 'Dagaare_dga', 'Mampruli_maw']

from huggingface_hub import snapshot_download
snapshot_download('ghananlpcommunity/ghana-speech', repo_type='dataset',
                  local_dir='data/parquet',
                  allow_patterns=[f'{c}/train-00000-*' for c in SMOKE_LANGS],
                  max_workers=8)
!du -sh data/parquet

In [ ]:
from kasa42.asr.train import train, TrainConfig

# batch_duration 120 for 16 GB; the H200 default is 320 and can go far higher.
ckpt = train(TrainConfig(smoke=True, languages=SMOKE_LANGS,
                         batch_duration=120.0, out_dir='checkpoints/smoke'))
print('checkpoint:', ckpt)

## 4 · Baselines — do they even load and decode?

Correctness of the plumbing, not the numbers. 30 steps of training predicts nothing.

In [ ]:
import torch
from transformers import AutoModelForCTC, AutoProcessor
from kasa42.asr.baselines import DONDO, DONDO_LANGS, iso_of
from kasa42.asr.dataset import decode_audio

proc = AutoProcessor.from_pretrained(DONDO)
m = AutoModelForCTC.from_pretrained(DONDO).cuda().eval()

import pyarrow.parquet as pq, glob
f = sorted(glob.glob('data/parquet/Kusaal_kus/*.parquet'))[0]
rows = next(pq.ParquetFile(f).iter_batches(batch_size=4)).to_pylist()
wavs = [decode_audio(r['audio']) for r in rows]

inp = proc(wavs, sampling_rate=16000, return_tensors='pt', padding=True).to('cuda')
with torch.no_grad():
    logits = m(**inp).logits
hyp = proc.batch_decode(logits.argmax(-1).cpu().numpy())
hyp = hyp.text if hasattr(hyp, 'text') else hyp
for r, h in zip(rows, hyp):
    print('REF', r['text'][:70]); print('DONDO', h[:70]); print()

## 5 · Export path — the demo must outlive the GPU

In [ ]:
!python -m kasa42.asr.export --checkpoint checkpoints/smoke/final.pt \
    --model-config checkpoints/smoke/config.json --out-dir export_smoke

# The check that matters: does it run with the GPU hidden?
!CUDA_VISIBLE_DEVICES='' KASA42_EXPORT=export_smoke KASA42_MODE=onnx python -c "\
import sys; sys.path.insert(0,'src'); import numpy as np; \
from kasa42.app.app import Engine; e=Engine('onnx'); \
t,l,c,dt=e.transcribe(16000, np.zeros(32000,dtype=np.float32)); \
print('mode', e.mode, '| lang', l, '|', f'{dt*1000:.0f}ms')"

## 6 · What did we learn?

Write down anything that failed and fix it in `src/`, then re-run from the top.
Do not carry a known failure into Thursday.

Checklist before closing this notebook:

- [ ] `test_text.py` and `test_pipeline.py` fully green
- [ ] `bench_gpu.py` gives an H200 estimate under ~12 h
- [ ] `train(smoke=True)` completes and the loss moves
- [ ] DONDO baseline produces plausible Kusaal text
- [ ] ONNX export runs **with `CUDA_VISIBLE_DEVICES=''`**
- [ ] Largest `batch_duration` that fits on 16 GB is recorded — the H200 has
      141 GB, so Thursday's value can be far larger

In [ ]:
import json, pathlib
p = pathlib.Path('results/bench_gpu.json')
if p.exists():
    print(json.dumps(json.loads(p.read_text()), indent=2))